In [1]:

import pandas as pd
from leviVSchub import *


In [2]:
summary = load_summary_intervals("summary_enriched.json")

df_intervals = intervals_to_dataframe(summary)

df_intervals.head()


,run,day,start_time,end_time,duration,leg_type,boat,name,mast,total_weight,master_leeward,avg_SOG,SOG_var,avg_TWS,avg_TWA,SOG_ref,pol_ratio,stability
0,25_11_2025_Run1,25_11_2025,1.764078e+09,1.764079e+09,65.001,upwind,1,Gian Stragiotti,Chub,112.1,False,21.081679,0.685094,20.236901,32.776496,22.866727,92.193690,0.959474
1,25_11_2025_Run1,25_11_2025,1.764078e+09,1.764079e+09,65.001,upwind,2,Max Maeder,Levi,111.5,True,21.322137,0.575966,20.222497,32.185617,22.864521,93.254248,0.959474
2,25_11_2025_Run1,25_11_2025,1.764079e+09,1.764079e+09,37.000,downwind,1,Gian Stragiotti,Chub,112.1,False,32.354667,1.023546,20.244787,-150.833800,30.059314,107.636077,0.938433
3,25_11_2025_Run1,25_11_2025,1.764079e+09,1.764079e+09,37.000,downwind,2,Max Maeder,Levi,111.5,True,32.810667,0.948195,20.252175,-154.113191,30.099028,109.009056,0.938433
4,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,68.515,upwind,1,Gian Stragiotti,Chub,112.1,True,21.044928,1.076835,24.750529,30.935558,22.860999,92.056028,0.943485


In [3]:
df_paired = build_paired_intervals(df_intervals)
df_paired.head(5)

,run,boat1_day,start_time,end_time,boat1_duration,leg_type,boat1_name,boat1_mast,boat1_total_weight,boat1_master_leeward,...,boat2_avg_TWS,boat2_avg_TWA,boat2_SOG_ref,boat2_pol_ratio,boat2_stability,d_pol_ratio,d_avg_SOG,d_avg_SOG_mast,d_weight_mast,levi_rider
0,25_11_2025_Run1,25_11_2025,1.764078e+09,1.764079e+09,65.001,upwind,Gian Stragiotti,Chub,112.1,False,...,20.222497,32.185617,22.864521,93.254248,0.959474,1.060557,0.240458,0.240458,-0.6,Max Maeder
1,25_11_2025_Run1,25_11_2025,1.764079e+09,1.764079e+09,37.000,downwind,Gian Stragiotti,Chub,112.1,False,...,20.252175,-154.113191,30.099028,109.009056,0.938433,1.372979,0.456000,0.456000,-0.6,Max Maeder
2,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,68.515,upwind,Gian Stragiotti,Chub,112.1,True,...,24.761937,30.601038,22.860261,94.737529,0.943485,2.681501,0.612319,0.612319,-0.6,Max Maeder
3,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,33.516,downwind,Gian Stragiotti,Chub,112.1,True,...,22.621696,-156.655570,30.157696,110.541483,0.941421,4.469691,1.394118,1.394118,-0.6,Max Maeder
4,25_11_2025_Run3,25_11_2025,1.764079e+09,1.764079e+09,64.483,upwind,Gian Stragiotti,Chub,112.1,False,...,21.631589,32.178469,22.864497,94.829584,0.953840,-1.758173,-0.403077,-0.403077,-0.6,Max Maeder


# I. First analysis: means and t-tests

In [4]:
df = df_intervals.copy()

leg_mast = pd.crosstab(df["leg_type"], df["mast"])
print("Leg-type exposure by mast:")
for mast in leg_mast.columns:
    total = leg_mast[mast].sum()
    for leg in leg_mast.index:
        n = leg_mast.loc[leg, mast]
        print(
            f"- {mast}: {n} intervals ({100*n/total:.3f}%) were {leg}."
        )
print()

rider_mast = pd.crosstab(df["name"], df["mast"])
print("Rider–mast exposure:")
for rider in rider_mast.index:
    total = rider_mast.loc[rider].sum()
    chub_n = rider_mast.loc[rider, "Chub"]
    levi_n = rider_mast.loc[rider, "Levi"]
    print(
        f"- {rider} sailed {chub_n} intervals on Chub ({100*chub_n/total:.1f}%) "
        f"and {levi_n} on Levi ({100*levi_n/total:.3f}%)."
    )
print()

mean_by_mast(df, "All legs")
mean_by_mast(df[df["leg_type"] == "upwind"], "Upwind only")
mean_by_mast(df[df["leg_type"] == "downwind"], "Downwind only")

Leg-type exposure by mast:
- Chub: 27 intervals (48.214%) were downwind.
- Chub: 29 intervals (51.786%) were upwind.
- Levi: 27 intervals (48.214%) were downwind.
- Levi: 29 intervals (51.786%) were upwind.

Rider–mast exposure:
- Gian Stragiotti sailed 31 intervals on Chub (55.4%) and 25 on Levi (44.643%).
- Max Maeder sailed 25 intervals on Chub (44.6%) and 31 on Levi (55.357%).

All legs
Average conditions — TWS: 16.576/16.577 kn, TWA: -60.538/-60.911°, Weight: 111.812/111.788 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): -0.142 kn. Chub is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = -0.15, p = 0.885
The difference in mean SOG is not statistically significant (p = 0.885 ≥ 0.05).
Upwind only
Average conditions — TWS: 16.874/16.874 kn, TWA: 30.229/29.318°, Weight: 111.814/111.786 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): 0.107 kn. Levi is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = 0.21, p = 0.

# II. OLS Regression, predicting SOG(Levi) − SOG(Chub) in order to reduce effect of external conditions

## 1) General OLS

In [5]:
# SOG model
results = analyze_mast_effect(df_paired)
print(results["sog_model"].summary())


                            OLS Regression Results                            
Dep. Variable:         d_avg_SOG_mast   R-squared:                       0.209
Model:                            OLS   Adj. R-squared:                  0.163
Method:                 Least Squares   F-statistic:                     5.482
Date:                Tue, 12 May 2026   Prob (F-statistic):            0.00238
Time:                        14:29:19   Log-Likelihood:                -95.549
No. Observations:                  56   AIC:                             199.1
Df Residuals:                      52   BIC:                             207.2
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

## 2) Upwind OLS

In [6]:
df_paired_upwind = df_paired[df_paired["leg_type"] == "upwind"]
model_upwind = analyze_mast_effect_by_leg(df_paired_upwind, "UPWIND ONLY")

Paired OLS — UPWIND ONLY
N paired intervals: 29
                            OLS Regression Results                            
Dep. Variable:         d_avg_SOG_mast   R-squared:                       0.100
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                    0.6380
Date:                Tue, 12 May 2026   Prob (F-statistic):              0.536
Time:                        14:29:19   Log-Likelihood:                -55.947
No. Observations:                  29   AIC:                             117.9
Df Residuals:                      26   BIC:                             122.0
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

## 2) Downwind OLS

In [7]:
df_paired_downwind = df_paired[df_paired["leg_type"] == "downwind"]
model_downwind = analyze_mast_effect_by_leg(df_paired_downwind, "DOWNWIND ONLY")

Paired OLS — DOWNWIND ONLY
N paired intervals: 27
                            OLS Regression Results                            
Dep. Variable:         d_avg_SOG_mast   R-squared:                       0.591
Model:                            OLS   Adj. R-squared:                  0.557
Method:                 Least Squares   F-statistic:                     15.49
Date:                Tue, 12 May 2026   Prob (F-statistic):           4.79e-05
Time:                        14:29:19   Log-Likelihood:                -28.793
No. Observations:                  27   AIC:                             63.59
Df Residuals:                      24   BIC:                             67.47
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------

# III. Rider-specific unpaired analysis
## 0 ) All riders

In [8]:
model_interaction = smf.ols(
    "avg_SOG ~ C(mast) * C(name) + C(leg_type) + avg_TWS + C(master_leeward)",
    data=df_intervals
).fit(cov_type="HC3")

print(model_interaction.summary())


                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.915
Method:                 Least Squares   F-statistic:                     319.6
Date:                Tue, 12 May 2026   Prob (F-statistic):           4.69e-65
Time:                        14:29:19   Log-Likelihood:                -201.16
No. Observations:                 112   AIC:                             416.3
Df Residuals:                     105   BIC:                             435.3
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

# 1) Gian 
## 1) a. Gian mean and t tests

In [9]:
df_gian = df_intervals[df_intervals["name"] == "Gian Stragiotti"]
df_gian.head()

,run,day,start_time,end_time,duration,leg_type,boat,name,mast,total_weight,master_leeward,avg_SOG,SOG_var,avg_TWS,avg_TWA,SOG_ref,pol_ratio,stability
0,25_11_2025_Run1,25_11_2025,1.764078e+09,1.764079e+09,65.001,upwind,1,Gian Stragiotti,Chub,112.1,False,21.081679,0.685094,20.236901,32.776496,22.866727,92.193690,0.959474
2,25_11_2025_Run1,25_11_2025,1.764079e+09,1.764079e+09,37.000,downwind,1,Gian Stragiotti,Chub,112.1,False,32.354667,1.023546,20.244787,-150.833800,30.059314,107.636077,0.938433
4,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,68.515,upwind,1,Gian Stragiotti,Chub,112.1,True,21.044928,1.076835,24.750529,30.935558,22.860999,92.056028,0.943485
6,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,33.516,downwind,1,Gian Stragiotti,Chub,112.1,True,31.942647,1.109971,22.628309,-154.847779,30.114177,106.071791,0.941421
8,25_11_2025_Run3,25_11_2025,1.764079e+09,1.764079e+09,64.483,upwind,1,Gian Stragiotti,Chub,112.1,False,22.085385,0.657233,21.633331,32.509592,22.865615,96.587758,0.953840


In [10]:
print(pd.crosstab(df_gian["leg_type"], df_gian["mast"], normalize="index") * 100)

mean_by_mast(df_gian, "Gian – all legs")

df_gian_up = df_gian[df_gian["leg_type"] == "upwind"].copy()
mean_by_mast(df_gian_up, "Gian – upwind")

df_gian_down = df_gian[df_gian["leg_type"] == "downwind"].copy()
mean_by_mast(df_gian_down, "Gian – downwind")

mast           Chub       Levi
leg_type                      
downwind  55.555556  44.444444
upwind    55.172414  44.827586
Gian – all legs
Average conditions — TWS: 17.043/16.200 kn, TWA: -59.195/-62.156°, Weight: 112.200/112.100 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): -0.960 kn. Chub is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = -0.70, p = 0.489
The difference in mean SOG is not statistically significant (p = 0.489 ≥ 0.05).
Gian – upwind
Average conditions — TWS: 17.171/16.632 kn, TWA: 32.125/27.546°, Weight: 112.200/112.100 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): -0.173 kn. Chub is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = -0.20, p = 0.844
The difference in mean SOG is not statistically significant (p = 0.844 ≥ 0.05).
Gian – downwind
Average conditions — TWS: 16.905/15.739 kn, TWA: -158.126/-157.839°, Weight: 112.200/112.100 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): -1.7


## 1) b. Gian OLS all legs

In [11]:
model_gian = smf.ols(
    "avg_SOG ~ C(mast) + C(leg_type) + avg_TWS + C(master_leeward)",
    data=df_gian
).fit(cov_type="HC3")
print(model_gian.summary())

                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.871
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     147.5
Date:                Tue, 12 May 2026   Prob (F-statistic):           2.28e-27
Time:                        14:29:19   Log-Likelihood:                -113.97
No. Observations:                  56   AIC:                             237.9
Df Residuals:                      51   BIC:                             248.1
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

Again, levi reduces the avg_SOG of 1 knot, and it is definitely statistically significant (p = 0.006)

## 1) b. Gian OLS upwind

In [12]:
model_gian_up = smf.ols(
    "avg_SOG ~ C(mast) + avg_TWS + C(master_leeward)",
    data=df_gian_up
).fit(cov_type="HC3")

print(model_gian_up.summary())


                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.078
Model:                            OLS   Adj. R-squared:                 -0.032
Method:                 Least Squares   F-statistic:                     3.350
Date:                Tue, 12 May 2026   Prob (F-statistic):             0.0350
Time:                        14:29:19   Log-Likelihood:                -65.136
No. Observations:                  29   AIC:                             138.3
Df Residuals:                      25   BIC:                             143.7
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

## 1) c. Gian OLS downwind

In [13]:
model_gian_down = smf.ols(
    "avg_SOG ~ C(mast) + avg_TWS + C(master_leeward)",
    data=df_gian_down
).fit(cov_type="HC3")

print(model_gian_down.summary())


                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.418
Model:                            OLS   Adj. R-squared:                  0.342
Method:                 Least Squares   F-statistic:                     7.328
Date:                Tue, 12 May 2026   Prob (F-statistic):            0.00128
Time:                        14:29:19   Log-Likelihood:                -39.057
No. Observations:                  27   AIC:                             86.11
Df Residuals:                      23   BIC:                             91.30
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

# 1) Max 
## 1) a. Max mean and t tests

In [14]:
df_max = df_intervals[df_intervals["name"] == "Max Maeder"]
df_max.head()

,run,day,start_time,end_time,duration,leg_type,boat,name,mast,total_weight,master_leeward,avg_SOG,SOG_var,avg_TWS,avg_TWA,SOG_ref,pol_ratio,stability
1,25_11_2025_Run1,25_11_2025,1.764078e+09,1.764079e+09,65.001,upwind,2,Max Maeder,Levi,111.5,True,21.322137,0.575966,20.222497,32.185617,22.864521,93.254248,0.959474
3,25_11_2025_Run1,25_11_2025,1.764079e+09,1.764079e+09,37.000,downwind,2,Max Maeder,Levi,111.5,True,32.810667,0.948195,20.252175,-154.113191,30.099028,109.009056,0.938433
5,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,68.515,upwind,2,Max Maeder,Levi,111.5,False,21.657246,0.738798,24.761937,30.601038,22.860261,94.737529,0.943485
7,25_11_2025_Run2,25_11_2025,1.764079e+09,1.764079e+09,33.516,downwind,2,Max Maeder,Levi,111.5,False,33.336765,0.725835,22.621696,-156.655570,30.157696,110.541483,0.941421
9,25_11_2025_Run3,25_11_2025,1.764079e+09,1.764079e+09,64.483,upwind,2,Max Maeder,Levi,111.5,True,21.682308,0.609418,21.631589,32.178469,22.864497,94.829584,0.953840


In [15]:
print(pd.crosstab(df_max["leg_type"], df_max["mast"], normalize="index") * 100)

mean_by_mast(df_max, "Max – all legs")

df_max_up = df_max[df_max["leg_type"] == "upwind"].copy()
mean_by_mast(df_max_up, "Max – upwind")

df_max_down = df_max[df_max["leg_type"] == "downwind"].copy()
mean_by_mast(df_max_down, "Max – downwind")

mast           Chub       Levi
leg_type                      
downwind  44.444444  55.555556
upwind    44.827586  55.172414
Max – all legs
Average conditions — TWS: 16.200/17.043 kn, TWA: -61.620/-59.367°, Weight: 111.500/111.400 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): 0.550 kn. Levi is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = 0.40, p = 0.693
The difference in mean SOG is not statistically significant (p = 0.693 ≥ 0.05).
Max – upwind
Average conditions — TWS: 16.632/17.171 kn, TWA: 28.689/31.500°, Weight: 111.500/111.400 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): 0.315 kn. Levi is faster on average in the raw data.
According to the Welch t-test on avg_SOG: t = 0.72, p = 0.480
The difference in mean SOG is not statistically significant (p = 0.480 ≥ 0.05).
Max – downwind
Average conditions — TWS: 15.739/16.905 kn, TWA: -157.950/-157.806°, Weight: 111.500/111.400 kg (Levi / Chub).
Mean SOG difference (Levi − Chub): 0.723 kn. L


## 1) b. Max OLS all legs

In [16]:
model_max = smf.ols(
    "avg_SOG ~ C(mast) + C(leg_type) + avg_TWS + C(master_leeward)",
    data=df_max
).fit(cov_type="HC3")  # robust SE
print(model_max.summary())

                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.969
Model:                            OLS   Adj. R-squared:                  0.967
Method:                 Least Squares   F-statistic:                     406.1
Date:                Tue, 12 May 2026   Prob (F-statistic):           5.47e-38
Time:                        14:29:19   Log-Likelihood:                -73.018
No. Observations:                  56   AIC:                             156.0
Df Residuals:                      51   BIC:                             166.2
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             


## 1) b. Max OLS upwind

In [17]:
model_max_up = smf.ols(
    "avg_SOG ~ C(mast) + avg_TWS + C(master_leeward)",
    data=df_max_up
).fit(cov_type="HC3")

print(model_max_up.summary())


                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.493
Model:                            OLS   Adj. R-squared:                  0.433
Method:                 Least Squares   F-statistic:                     6.354
Date:                Tue, 12 May 2026   Prob (F-statistic):            0.00237
Time:                        14:29:19   Log-Likelihood:                -34.046
No. Observations:                  29   AIC:                             76.09
Df Residuals:                      25   BIC:                             81.56
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             


## 1) b. Max OLS downwind

In [18]:
model_max_down = smf.ols(
    "avg_SOG ~ C(mast) + avg_TWS + C(master_leeward)",
    data=df_max_down
).fit(cov_type="HC3")

print(model_max_down.summary())


                            OLS Regression Results                            
Dep. Variable:                avg_SOG   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     2.446
Date:                Tue, 12 May 2026   Prob (F-statistic):             0.0896
Time:                        14:29:19   Log-Likelihood:                -30.832
No. Observations:                  27   AIC:                             69.66
Df Residuals:                      23   BIC:                             74.85
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             